# Gravitational-Wave Capture Walkthrough

This notebook recomputes a coarse version of the paper's neutron-star capture grid with Captus. The reference system is a $1.4\,M_\odot$, 11 km neutron star at 8 kpc. The default grid is intentionally small. It illustrates the trends without claiming a pixel-level reproduction of the manuscript figure.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy import constants as const
from matplotlib.colors import LogNorm
from tdpy.verbosity import print

import captus
from captus.gw_capture.simulation import GWCapture

plt.rcParams["text.usetex"] = False
print(f"Python {sys.version.split()[0]} | Captus {captus.__version__}")

In [ ]:
grid_size = 8
masses_msun = np.logspace(-8, 2, grid_size)
velocities_kms = np.logspace(0, 3, grid_size)

capture_model = GWCapture(
    {
        "name": "Typical_Neutron_Star",
        "M_star": 1.4 * const.M_sun.value,  # [kg]
        "R_star": 11e3,  # [m]
        "gridsize": grid_size,
        "M_pbh_values": masses_msun * const.M_sun.value,  # [kg]
        "v_inf_values": velocities_kms * 1e3,  # [m/s]
        "r": 8.0,  # [kpc]
    }
)
print(f"Grid: {grid_size} masses x {grid_size} velocities")


## Calculate the paper quantities

Captus evaluates the capture cross-section and post-capture coalescence time on this mass-velocity grid. If `PBHBOUNDS_PATH` is set to the Captus-compatible PBHbounds checkout, the equilibrium-count calculation also evaluates the observationally bound PBH fraction; otherwise that optional panel is skipped.

In [ ]:
cross_section_grid = capture_model.get_capture_cross_section()
coalescence_time_grid = capture_model.get_coalescence_time()

neq_grid = None
if os.environ.get("PBHBOUNDS_PATH"):
    neq_grid = capture_model.get_systems_in_eq()
else:
    print("PBHBOUNDS_PATH is unset; equilibrium abundance will be omitted.")

expected_shape = (grid_size, grid_size)
np.testing.assert_equal(cross_section_grid.shape, expected_shape)
np.testing.assert_equal(coalescence_time_grid.shape, expected_shape)
print("Cross-section grid shape:", cross_section_grid.shape)
print("Positive cross-section cells:", np.count_nonzero(cross_section_grid > 0))


In [ ]:
def plot_positive_grid(axis, values, colorbar_label):
    values = np.asarray(values, dtype=float)
    valid = np.isfinite(values) & (values > 0)
    if not np.any(valid):
        axis.text(0.5, 0.5, "No positive values", ha="center", va="center")
        return

    lower = float(values[valid].min())
    upper = float(values[valid].max())
    if lower == upper:
        upper = lower * 10
    image = axis.pcolormesh(
        masses_msun,
        velocities_kms,
        np.ma.masked_where(~valid, values),
        shading="auto",
        cmap="magma",
        norm=LogNorm(vmin=lower, vmax=upper),
    )
    axis.set_xscale("log")
    axis.set_yscale("log")
    axis.set_xlabel(r"PBH mass [$M_\\odot$]")
    axis.set_ylabel(r"$v_\\infty$ [km s$^{-1}$]")
    axis.axhline(550, color="cyan", linestyle="--", linewidth=1)  # [km/s]
    axis.grid(False)
    axis.figure.colorbar(image, ax=axis).set_label(colorbar_label)


grid_values = [cross_section_grid, coalescence_time_grid]
grid_titles = ["GW capture cross-section", "Post-capture lifetime"]
grid_labels = [r"$\\sigma_{\\rm cap}$ [AU$^2$]", "Coalescence time [yr]"]
if neq_grid is not None:
    grid_values.append(neq_grid)
    grid_titles.append("Equilibrium abundance")
    grid_labels.append(r"Bound $N_{\\rm eq}$ contribution")

figure, axes = plt.subplots(
    1, len(grid_values), figsize=(4.5 * len(grid_values), 4), layout="constrained"
)
for axis, values, title, label in zip(axes, grid_values, grid_titles, grid_labels):
    plot_positive_grid(axis, values, label)
    axis.set_title(title)
figure.suptitle("Typical neutron star: 1.4 solar masses, 11 km, r = 8 kpc")

gw_output_dir = Path("paper_examples")
gw_output_dir.mkdir(parents=True, exist_ok=True)
gw_output_path = gw_output_dir / "gw_capture_neutron_star.png"
print(f"Writing to {gw_output_path}...")
figure.savefig(gw_output_path, dpi=180)
display(figure)
plt.close(figure)


## Interpret and export

The manuscript finds that neutron-star gravitational-wave capture is concentrated toward slow incoming speeds and PBH masses comparable to or above the neutron-star mass. The post-capture inspiral times are short, leaving a very small equilibrium population. Its discussion reports $N_{\\rm eq}(f_{\\rm bound}) \\lesssim 10^{-12}$ in the Solar vicinity. The coarse grid below illustrates the trends, while the manuscript's 100-by-100 grid and bound-fraction calculation require the PBHbounds data path.

The figure is written beneath `paper_examples/` relative to the notebook's working directory. Export with `jupyter nbconvert --to html examples/paper_gw_capture.ipynb` from the repository root.